# AutoService-LLM: baseline evaluation (Step 1 of 3)
Compares **base** vs **prompted** `Qwen2.5-1.5B-Instruct` on the 47 held-out test questions.
Runtime: Runtime > Change runtime type > **T4 GPU**, then Run all (about 5-10 min).

In [ ]:
!nvidia-smi -L
!pip -q install transformers accelerate rouge-score

In [ ]:
import json, random, time, os
!git clone -q https://github.com/Nisha-Lohar04/auto-service-llm.git

def load(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
train = load("auto-service-llm/data/processed/train.jsonl")
test  = load("auto-service-llm/data/processed/test.jsonl")
print("train:", len(train), "test:", len(test))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("loaded", MODEL)

In [ ]:
SYSTEM = ("You are an automotive service assistant. Answer car maintenance questions accurately in 1-3 sentences. "
          "Prioritise safety, mention the owner's manual when specifications vary, and say when a professional inspection is needed.")

random.seed(42)
shots = random.sample([r for r in train if r["origin"] == "claude_authored"], 3)  # 3 fixed few-shot examples from TRAIN only

def build_messages(q, mode):
    if mode == "base":                       # no custom prompt
        return [{"role": "user", "content": q}]
    msgs = [{"role": "system", "content": SYSTEM}]   # system prompt + 3 examples
    for s in shots:
        msgs += [{"role": "user", "content": s["question"]},
                 {"role": "assistant", "content": s["answer"]}]
    msgs.append({"role": "user", "content": q})
    return msgs

@torch.no_grad()
def generate(q, mode):
    text = tok.apply_chat_template(build_messages(q, mode), tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    dt = time.time() - t0
    ans = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return ans, dt

print(generate(test[0]["question"], "base"))

In [ ]:
import pandas as pd
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rows = []
for mode in ["base", "prompted"]:
    for r in test:
        ans, dt = generate(r["question"], mode)
        rows.append({"mode": mode, "example_id": r["example_id"], "group_id": r["group_id"],
                     "origin": r["origin"], "question": r["question"], "reference": r["answer"],
                     "prediction": ans, "rougeL": scorer.score(r["answer"], ans)["rougeL"].fmeasure,
                     "latency_s": dt, "words": len(ans.split())})
    print(mode, "done")

df = pd.DataFrame(rows)
summary = df.groupby("mode")[["rougeL", "latency_s", "words"]].mean().round(3)
summary

In [ ]:
for mode in ["base", "prompted"]:
    print("=====", mode)
    for _, r in df[df["mode"] == mode].head(3).iterrows():
        print("Q:", r["question"]); print("REF:", r["reference"]); print("PRED:", r["prediction"]); print()

In [ ]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/baseline_predictions.csv", index=False)
summary.to_csv("results/baseline_summary.csv")
from google.colab import files
files.download("results/baseline_predictions.csv")
files.download("results/baseline_summary.csv")